# 13 · Reprodução de Lira et al.: tabelas e diferença com × sem vazamento

Lê `resultados/reproducao_lira/` e grava em `resultados/reproducao_lira_analise/`:

* `tabela4_reproduzida.csv` / `tabela5_reproduzida.csv`: receita de Lira na partição **com** vazamento (formato das Tabelas 4 e 5: mesmas linhas e 16 colunas);
* `tabela4_sem_vazamento.csv` / `tabela5_sem_vazamento.csv`: tudo igual, partição **agrupada por caso**;
* `tabela4_diferenca.csv` / `tabela5_diferenca.csv`: com vazamento − sem vazamento;
* `reproducao_vs_publicado.csv`: cada linha reproduzida ao lado da publicada;
* `resumo_reproducao.json`: partições, taxa de vazamento, erros médios e correlação de postos.

Não precisa de GPU. As métricas saem das **classes previstas**, como nas tabelas do artigo.

In [ ]:
DIR_DRIVE = "/content/drive/MyDrive/Colab Notebooks/tcc-final"
SHA_TABELAS_ESPERADO = "74cb1aedf91057c2cbf92896d02759a6d992bf7962e2ee18eb07f2b16d616d7c"
PARTICAO_VAZADA, PARTICAO_AGRUPADA = "vazado_p42", "agrupado_d0"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import sys, json
from pathlib import Path
import numpy as np, pandas as pd
sys.path.insert(0, DIR_DRIVE)
import protocolo_comum as pc
pc.checar_sentinela_drive(DIR_DRIVE)
R = Path(DIR_DRIVE) / "resultados" / "reproducao_lira"
SAIDA = Path(DIR_DRIVE) / "resultados" / "reproducao_lira_analise"; SAIDA.mkdir(exist_ok=True)
dec = json.loads((Path(DIR_DRIVE) / "resultados" / "calibracao_lira.json").read_text())
ARQ_TAB = Path(DIR_DRIVE) / "dados" / "lira2024_tabelas_completas.csv"
assert pc.sha256_arquivo(ARQ_TAB) == SHA_TABELAS_ESPERADO
PUB = pd.read_csv(ARQ_TAB)
METRICAS = ["acc", "prec_M", "prec_m", "prec_W", "rec_M", "rec_m", "rec_W", "f1_M", "f1_m", "f1_W",
            "auc_M", "auc_m", "auc_W", "espec_M", "espec_m", "espec_W"]
linhas, brutos = [], {}
for arq in sorted(R.glob(f"{dec['receita']}/*/*/*/resultado.json")):
    r = json.loads(arq.read_text())
    if r["teste_rapido"] and not (dec["receita"] != "A" and r["epocas"] == dec["epocas_B"]):
        continue
    brutos[(r["particao"], r["tarefa"], r["modelo"])] = r
    linhas.append({"particao": r["particao"], "tarefa": r["tarefa"], "modelo_leguwoi": r["modelo"], **r["metricas"],
                   "treino_min": r["treino_min"], "lote": r["lote"], "desvios": "; ".join(r["desvios"]),
                   "n_treino": r["dados"]["n_treino"], "n_teste": len(r["y_teste"])})
res = pd.DataFrame(linhas)
print("Receita:", dec["receita"], "| unidades:", len(res), "|", res.groupby(["tarefa", "particao"]).size().to_dict())

In [ ]:
def tabela(tarefa, particao):
    base = PUB[PUB["tarefa"] == tarefa][["modelo_leguwoi", "rotulo_artigo", "torchvision"]]
    t = base.merge(res[(res["tarefa"] == tarefa) & (res["particao"] == particao)][["modelo_leguwoi"] + METRICAS],
                   on="modelo_leguwoi", how="left")
    return t   # mesma ordem de linhas do artigo; NaN = unidade ainda não rodada

resumo = {"receita": dec["receita"], "precisao": dec["precisao"], "particoes": [PARTICAO_VAZADA, PARTICAO_AGRUPADA], "por_tarefa": {}}
comparacoes = []
for tarefa, n in (("es", 4), ("dist", 5)):
    vaz, agr = tabela(tarefa, PARTICAO_VAZADA), tabela(tarefa, PARTICAO_AGRUPADA)
    dif = vaz.copy(); dif[METRICAS] = vaz[METRICAS] - agr[METRICAS]
    for nome, t in ((f"tabela{n}_reproduzida", vaz), (f"tabela{n}_sem_vazamento", agr), (f"tabela{n}_diferenca", dif)):
        t.round(4).to_csv(SAIDA / f"{nome}.csv", index=False)
        t.drop(columns=["modelo_leguwoi", "torchvision"]).to_latex(SAIDA / f"{nome}.tex", index=False, float_format="%.3f", na_rep="--")
    pub = PUB[PUB["tarefa"] == tarefa].reset_index(drop=True)
    comp = pub[["tarefa", "modelo_leguwoi", "rotulo_artigo"]].copy()
    for c in ("acc", "f1_M", "auc_M"):
        comp[f"pub_{c}"], comp[f"vazado_{c}"], comp[f"agrupado_{c}"] = pub[c], vaz[c], agr[c]
    comparacoes.append(comp)
    ok = vaz["acc"].notna() & agr["acc"].notna()
    resumo["por_tarefa"][tarefa] = {
        "linhas_completas": int(ok.sum()),
        "erro_abs_medio_vs_publicado": {c: float((vaz.loc[ok, c] - pub.loc[ok, c]).abs().mean()) for c in ("acc", "f1_M", "auc_M")},
        "spearman_acc_reproduzida_x_publicada": float(vaz.loc[ok, "acc"].corr(pub.loc[ok, "acc"], method="spearman")),
        "media_com_vazamento": {c: float(vaz.loc[ok, c].mean()) for c in ("acc", "f1_M", "auc_M")},
        "media_sem_vazamento": {c: float(agr.loc[ok, c].mean()) for c in ("acc", "f1_M", "auc_M")},
        "queda_media": {c: float(dif.loc[ok, c].mean()) for c in ("acc", "f1_M", "auc_M")},
        "modelos_que_pioram_sem_vazamento_f1_M": int((dif.loc[ok, "f1_M"] > 0).sum()),
        "classes_teste": {p: brutos[k]["dados"]["classes_teste"] for p in (PARTICAO_VAZADA, PARTICAO_AGRUPADA)
                          for k in [next((k for k in brutos if k[0] == p and k[1] == tarefa), None)] if k},
    }
    print(f"Tabela {n} ({tarefa}):", json.dumps(resumo["por_tarefa"][tarefa], indent=1))
    display(comp.round(3))
pd.concat(comparacoes).round(4).to_csv(SAIDA / "reproducao_vs_publicado.csv", index=False)
resumo["horas_de_treino"] = float(res["treino_min"].sum() / 60)
resumo["desvios"] = sorted(set(d for d in res["desvios"] if d))
pc.gravar_json_atomico(resumo, SAIDA / "resumo_reproducao.json")
print("Gravado em", SAIDA)

**Como ler.** Uma única partição por lado: a diferença de um modelo isolado é ruidosa (várias redes treinadas do zero colapsam na classe majoritária, e aí a partição quase não importa). O que a comparação sustenta é o padrão no conjunto das linhas e nas ResNets pré-treinadas, que são as que de fato aprendem. Para barras de erro, rode o 12 de novo com `PARTICOES = ("vazado_p43", "agrupado_d1")` e `("vazado_p44", "agrupado_d2")`.